# ADMET-Engine v3.0 (*CardioPhys-3D*) — WP1 & WP2 Colab GPU Worker

Stateless Google Colab Pro worker for **ADMET-Engine v3.0** (`github.com/JakobFerschin/Admet-Engine`):
1. **WP1 (Multi-Channel ChEMBL v37 + pH 7.4 3D Conformers):** Streams censored (`=`, `>`, `<`) bioactivities for **hERG** (`CHEMBL240`), **Cav1.2** (`CHEMBL1940`), and **Nav1.5** (`CHEMBL1980`), enforces a zero-overlap Bemis–Murcko scaffold split, protonates molecules at physiological pH 7.4, and generates MMFF94s Boltzmann-weighted 3D conformer ensembles in parallel via `joblib`.
2. **WP2 (Differentiable FDA-CiPA Cardiomyocyte ODE on GPU):** Runs all 28 official FDA-CiPA reference drugs ($28 \times 4 = 112$ action potential trajectories) in parallel on CUDA GPU, computes `qNet` & `APD90`, proves multi-channel rescue on **Verapamil** & **Ranolazine**, and plots publication-grade action potential waveforms.

> **Runtime tip:** Select `Runtime -> Change runtime type -> T4 / L4 / A100 GPU` (High-RAM recommended for full 3D conformer generation).

In [ ]:
# @title 1. Clone / Sync Repo & Install Dependencies
import os, subprocess, sys, torch

REPO_DIR = '/content/Admet-Engine'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/JakobFerschin/Admet-Engine.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!pip install -q rdkit torch-geometric joblib scikit-learn requests matplotlib

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | PyTorch: {torch.__version__} | CPUs: {os.cpu_count()}')
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# @title 2. WP2 — GPU-Vectorized FDA-CiPA 28 Cardiomyocyte Validation
import sys, time
sys.path.insert(0, '/content/Admet-Engine/src')

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from physiology import CardiomyocyteSimulator, ChannelBlockProfile, CIPA_28_DRUGS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sim = CardiomyocyteSimulator(dt_ms=0.2, n_beats=2).to(device)

def _t(vals):
    return torch.tensor([1e6 if v is None else float(v) for v in vals], dtype=torch.float32, device=device)

t0 = time.perf_counter()
out = sim.simulate_cipa_protocol(
    cmax_free_uM=_t([d.cmax_free_uM for d in CIPA_28_DRUGS]),
    ic50_herg_uM=_t([d.ic50_herg_uM for d in CIPA_28_DRUGS]),
    ic50_cav12_uM=_t([d.ic50_cav12_uM for d in CIPA_28_DRUGS]),
    ic50_nav15_late_uM=_t([d.ic50_nav15_late_uM for d in CIPA_28_DRUGS]),
    ic50_nav15_peak_uM=_t([d.ic50_nav15_peak_uM for d in CIPA_28_DRUGS]),
    h_herg=_t([d.h_herg for d in CIPA_28_DRUGS]),
    h_cav12=_t([d.h_cav12 for d in CIPA_28_DRUGS]),
    h_nav15_late=_t([d.h_nav15_late for d in CIPA_28_DRUGS]),
    h_nav15_peak=_t([d.h_nav15_peak for d in CIPA_28_DRUGS]),
)
if device.type == 'cuda':
    torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

qnets = out['mean_qnet'].detach().cpu().numpy()
apds = out['apd90_1x_ms'].detach().cpu().numpy()
risks = [d.tdp_risk for d in CIPA_28_DRUGS]

y_hi_int = [1 if r in ('High', 'Intermediate') else 0 for r in risks]
y_hi_lo = [1 if r == 'High' else 0 for r in risks if r != 'Intermediate']
q_hi_lo = [-q for q, r in zip(qnets, risks) if r != 'Intermediate']
herg_ratio = [d.cmax_free_uM / (d.ic50_herg_uM or 1e6) for d in CIPA_28_DRUGS]
herg_hi_lo = [hr for hr, r in zip(herg_ratio, risks) if r != 'Intermediate']

print(f'Simulated 112 cardiomyocyte ODE trajectories on {device} in {elapsed:.2f}s!')
print(f'CiPA-ODE (-qNet) ROC-AUC (High+Int vs Low): {roc_auc_score(y_hi_int, -qnets):.4f}  (hERG-only: {roc_auc_score(y_hi_int, herg_ratio):.4f})')
print(f'CiPA-ODE (-qNet) ROC-AUC (High vs Low):     {roc_auc_score(y_hi_lo, q_hi_lo):.4f}  (hERG-only: {roc_auc_score(y_hi_lo, herg_hi_lo):.4f})')

In [ ]:
# @title 3. WP2 Figure — Verapamil Multi-Channel Rescue vs. Dofetilide Action Potentials
conc = torch.tensor([0.0, 0.008, 0.324, 0.324], device=device)  # Control, Dofetilide 4x, Verapamil hERG-only 4x, Verapamil Full 4x
herg = torch.tensor([1e6, 0.0049, 0.499, 0.499], device=device)
cav12 = torch.tensor([1e6, 1e6, 1e6, 0.202], device=device)

res = sim.simulate(
    ChannelBlockProfile.from_ic50_uM(conc_uM=conc, ic50_herg_uM=herg, ic50_cav12_uM=cav12),
    return_traces=True,
    trace_stride=2,
)

t = res.time_ms.cpu().numpy()
v = res.v_trace.cpu().numpy()
labels = [
    f'Control (qNet={res.qnet[0]:.4f}, APD90={res.apd90_ms[0]:.0f} ms)',
    f'Dofetilide 4x Cmax (High TdP; qNet={res.qnet[1]:.4f}, APD90={res.apd90_ms[1]:.0f} ms)',
    f'Verapamil hERG-only (False Positive; qNet={res.qnet[2]:.4f}, APD90={res.apd90_ms[2]:.0f} ms)',
    f'Verapamil Full hERG+Cav1.2 (Rescued Safe; qNet={res.qnet[3]:.4f}, APD90={res.apd90_ms[3]:.0f} ms)',
]
colors = ['#64748B', '#A51C30', '#B45309', '#0F766E']
styles = ['--', '-', ':', '-']

plt.figure(figsize=(9, 4.5), dpi=150)
for i in range(4):
    plt.plot(t, v[i], label=labels[i], color=colors[i], ls=styles[i], lw=2.0)
plt.xlim(0, 800)
plt.xlabel('Time (ms)')
plt.ylabel('Membrane Potential $V_m$ (mV)')
plt.title('CardioPhys-3D Mechanistic Proof: Verapamil Multi-Channel Rescue (ORd-CiPA)')
plt.grid(alpha=0.3)
plt.legend(fontsize=8.5, frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# @title 4. WP1 — Extract Multi-Channel ChEMBL v37 (hERG, Cav1.2, Nav1.5) with Tobit Censored Bounds
# Set MAX_PER_TARGET = None for the full ChEMBL corpus (~25k-30k rows), or 2000 for a quick smoke test.
MAX_PER_TARGET = None  # @param {type:'raw'}

from data.multichannel_extractor import extract_from_rest_api, build_multichannel_dataset

raw_rows = extract_from_rest_api(max_per_target=MAX_PER_TARGET)
df_mc = build_multichannel_dataset(raw_rows)

OUT_CSV = '/content/Admet-Engine/data/processed/cardiophys_multichannel_v3.csv'
df_mc.to_csv(OUT_CSV, index=False)
print(f'Saved {len(df_mc):,} compounds to {OUT_CSV}')
df_mc.head()

In [ ]:
# @title 5. WP1 — Parallel pH 7.4 Protonation + ETKDGv3/MMFF94s 3D Conformer Generation
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from data.conformer_3d import generate_3d_ensemble

NUM_CONFS = 5  # @param {type:'integer'} (use 10 on High-RAM Colab for final WP3 training)
N_JOBS = -1    # Use all Colab CPU cores

smiles_list = df_mc['canonical_smiles'].tolist()

def _featurize_one(smi):
    ens = generate_3d_ensemble(smi, num_confs=NUM_CONFS, protonate=True)
    return None if ens is None else ens.to_torch_dict()

results_3d = Parallel(n_jobs=N_JOBS, prefer='processes')(
    delayed(_featurize_one)(smi) for smi in tqdm(smiles_list, desc='3D Conformers (pH 7.4)')
)
valid_3d = {smi: r for smi, r in zip(smiles_list, results_3d) if r is not None}
OUT_PT = '/content/Admet-Engine/data/processed/cardiophys_3d_conformers_v3.pt'
torch.save(valid_3d, OUT_PT)
print(f'Saved {len(valid_3d):,} / {len(smiles_list):,} 3D conformer ensembles to {OUT_PT}')